# Step 8: Department & Role Analysis

This notebook performs recruitment analytics to:
1. Calculate department-wise candidate drop-off rates and compare them against company averages.
2. Drill down into specific roles within departments to identify role-specific drop-off rates.
3. Highlight significant differences (deltas) to identify process bottlenecks.

In [ ]:
import os
import pandas as pd
import numpy as np

# Path setup
FEATURES_PATH = os.path.join("..", "data", "processed", "candidate_features.csv")
print(f"Checking features file at: {FEATURES_PATH}")

## Load Data

We load the engineered candidate features dataset.

In [ ]:
df = pd.read_csv(FEATURES_PATH)
df.head()

## Part 1: Department-wise Drop-off Rates

We calculate total candidates, dropped candidates, joined candidates, and drop-off rate for each department, and calculate their delta relative to the overall company drop-off rate.

In [ ]:
# Company averages
company_total = len(df)
company_dropped = df['dropped'].sum()
company_dropoff_rate = (company_dropped / company_total) * 100 if company_total > 0 else 0.0

print(f"Company Total Candidates: {company_total}")
print(f"Company Dropped Candidates: {company_dropped}")
print(f"Company Overall Drop-off Rate: {company_dropoff_rate:.2f}%")

# Department stats
dept_stats = []
for dept_name, group in df.groupby('department'):
    total = len(group)
    dropped = group['dropped'].sum()
    joined = group['joined'].sum()
    rate = (dropped / total) * 100 if total > 0 else 0.0
    delta = rate - company_dropoff_rate
    dept_stats.append({
        'department': dept_name,
        'total_candidates': total,
        'dropped_candidates': dropped,
        'joined_candidates': joined,
        'dropoff_rate (%)': round(rate, 2),
        'delta_from_company_average (%)': round(delta, 2)
    })

df_dept = pd.DataFrame(dept_stats)
df_dept

## Part 2: Role-level Drilldown Analysis

We drill down to specific roles within departments and calculate their drop-off rates relative to their respective parent department's average drop-off rate.

In [ ]:
role_stats = []
for (dept_name, role_name), group in df.groupby(['department', 'role']):
    total = len(group)
    dropped = group['dropped'].sum()
    joined = group['joined'].sum()
    rate = (dropped / total) * 100 if total > 0 else 0.0
    
    # Get parent department average
    dept_avg = df_dept.loc[df_dept['department'] == dept_name, 'dropoff_rate (%)'].values[0]
    delta = rate - dept_avg
    
    role_stats.append({
        'department': dept_name,
        'role': role_name,
        'total_candidates': total,
        'dropped_candidates': dropped,
        'joined_candidates': joined,
        'role_dropoff_rate (%)': round(rate, 2),
        'department_average_dropoff_rate (%)': round(dept_avg, 2),
        'delta_from_department_average (%)': round(delta, 2)
    })

df_role = pd.DataFrame(role_stats).sort_values(by=['department', 'role_dropoff_rate (%)'], ascending=[True, False])
df_role